In [1]:
import logging
import sys
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# Ensure stdout logging prints directly to notebook/terminal
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

# -------------------------------------------------------------------------
# STEP 1: Load Data & Preprocess Features
# -------------------------------------------------------------------------
print("--> Loading dataset for Task 3...")
df = pd.read_csv("Metro_Interstate_Traffic_Volume.csv")
df["date_time"] = pd.to_datetime(df["date_time"])
df = df.sort_values("date_time").reset_index(drop=True)

# Engineer temporal features
df["hour"] = df["date_time"].dt.hour
df["day_of_week"] = df["date_time"].dt.dayofweek
df["month"] = df["date_time"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

# Cyclical trigonometric encodings
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24.0)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24.0)
df["day_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7.0)
df["day_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7.0)

# Select feature matrix and regression target
feature_cols = [
    "temp",
    "rain_1h",
    "snow_1h",
    "clouds_all",
    "hour_sin",
    "hour_cos",
    "day_sin",
    "day_cos",
    "is_weekend",
    "month",
]

X = df[feature_cols]
y = df["traffic_volume"]

# Split chronologically into Train (80%) and Test (20%) sets
split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Feature scaling required for Neural Network gradient optimization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(
    f"--> Dataset ready: {len(X_train)} train rows, {len(X_test)} test rows."
)

# -------------------------------------------------------------------------
# STEP 2: Train Neural Network Model (Demand Prediction)
# -------------------------------------------------------------------------
print("--> Training Multi-Layer Perceptron (MLP) Neural Network...")
mlp_model = MLPRegressor(
    hidden_layer_sizes=(32, 16),
    activation="relu",
    solver="adam",
    max_iter=30,  # Reduced iterations for fast execution
    random_state=42,
)
mlp_model.fit(X_train_scaled, y_train)

y_pred_mlp = mlp_model.predict(X_test_scaled)
mlp_mae = mean_absolute_error(y_test, y_pred_mlp)
mlp_r2 = r2_score(y_test, y_pred_mlp)

# -------------------------------------------------------------------------
# STEP 3: Train Surrogate Model & Explainability Analysis
# -------------------------------------------------------------------------
print("--> Training surrogate Random Forest Regressor...")
surrogate_rf = RandomForestRegressor(
    n_estimators=50, max_depth=10, random_state=42, n_jobs=-1
)
surrogate_rf.fit(X_train, y_train)

y_pred_rf = surrogate_rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)

print("--> Computing Permutation Feature Importance...")
sample_idx = np.random.choice(X_test.shape[0], 300, replace=False)
perm_importance = permutation_importance(
    surrogate_rf,
    X_test.iloc[sample_idx],
    y_test.iloc[sample_idx],
    n_repeats=3,
    random_state=42,
)

# -------------------------------------------------------------------------
# STEP 4: Print Output Tables Directly
# -------------------------------------------------------------------------
eval_df = pd.DataFrame(
    {
        "Model": [
            "MLP Neural Network (Primary)",
            "Random Forest (Surrogate Explainer)",
        ],
        "MAE (vehicles/hr)": [round(mlp_mae, 2), round(rf_mae, 2)],
        "R-Squared (R²)": [round(mlp_r2, 4), round(rf_r2, 4)],
    }
)

feature_importance = pd.DataFrame(
    {
        "Feature": feature_cols,
        "Importance_Score": perm_importance.importances_mean,
    }
).sort_values(by="Importance_Score", ascending=False)

print("\n" + "=" * 65)
print("TASK 3: NEURAL NETWORK DEMAND PREDICTION EVALUATION")
print("=" * 65)
print(eval_df.to_string(index=False))

print("\n" + "=" * 65)
print("TOP 5 FEATURE DRIVERS VIA EXPLAINABILITY ANALYSIS")
print("=" * 65)
print(feature_importance.head(5).to_string(index=False))


--> Loading dataset for Task 3...
--> Dataset ready: 38563 train rows, 9641 test rows.
--> Training Multi-Layer Perceptron (MLP) Neural Network...


C:\Users\vimih\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


--> Training surrogate Random Forest Regressor...
--> Computing Permutation Feature Importance...

TASK 3: NEURAL NETWORK DEMAND PREDICTION EVALUATION
                              Model  MAE (vehicles/hr)  R-Squared (R²)
       MLP Neural Network (Primary)             595.44          0.8322
Random Forest (Surrogate Explainer)             293.69          0.9267

TOP 5 FEATURE DRIVERS VIA EXPLAINABILITY ANALYSIS
   Feature  Importance_Score
  hour_cos          1.460406
  hour_sin          0.297412
   day_sin          0.070775
is_weekend          0.055194
   day_cos          0.010880
